# Plot DynEdge MC/BS resultater (part2)

RESULTS_CSV = BASE_DIR / 'results' / 'dynedge_mc_bs_fast_results.csv'
- histogram af `is_mc_pred` for BS vs MC
- ROC-kurve og AUC
- confusion matrix ved threshold 0.5

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

BASE_DIR = Path('/groups/icecube/holgerkc/Thesis_Analysis/MC_vs_BS_analysis')

# Find den nyeste results-fil automatisk
results_files = sorted((BASE_DIR / 'results').glob('dynedge_mc_bs_*_results.csv'), key=lambda p: p.stat().st_mtime)
if not results_files:
    raise FileNotFoundError('Ingen results CSV fundet i results/')
RESULTS_CSV = results_files[-1]

# Tilsvarende metrics JSON
METRICS_JSON = Path(str(RESULTS_CSV).replace('_results.csv', '_metrics.json'))

print('Results file:', RESULTS_CSV)
print('Exists:', RESULTS_CSV.exists())
print('Metrics file:', METRICS_JSON)
print('Exists:', METRICS_JSON.exists())

In [ ]:
results = pd.read_csv(RESULTS_CSV)
print('Shape:', results.shape)
print('Columns:', list(results.columns))
results.head()

In [ ]:
if METRICS_JSON.exists():
    metrics = json.loads(METRICS_JSON.read_text())
    print('Saved metrics:')
    print(json.dumps(metrics, indent=2))
else:
    print('No metrics json found.')

In [ ]:
required_cols = {'is_mc', 'is_mc_pred'}
missing = required_cols - set(results.columns)
if missing:
    raise ValueError(f'Missing required columns in results: {missing}')

y_true = results['is_mc'].astype(int).to_numpy()
y_score = results['is_mc_pred'].astype(float).to_numpy()
y_pred = (y_score >= 0.5).astype(int)

fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
acc = (y_true == y_pred).mean()

print(f'n_test = {len(y_true)}')
print(f'AUC = {roc_auc:.4f}')
print(f'Accuracy@0.5 = {acc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

n_mc    = metrics.get('n_mc', '?')
n_bs    = metrics.get('n_bs', '?')
n_test  = metrics.get('n_test', len(y_true))
epochs  = metrics.get('epochs', '?')
model_info = f"DynEdge  |  {n_mc} MC + {n_bs} BS muons (L3)  |  {epochs} epochs  \n  test size={n_test}"

# Histogram of model score P(MC) for BS vs MC
bins = np.linspace(0, 1, 41)
axes[0].hist(y_score[y_true == 0], bins=bins, histtype='step', linewidth=1.8, label='BS (y=0)')
axes[0].hist(y_score[y_true == 1], bins=bins, histtype='step', linewidth=1.8, label='MC (y=1)')
axes[0].set_xlabel('Model score: P(MC)')
axes[0].set_ylabel('Number of events')
axes[0].set_yscale('log')
axes[0].set_title(f'Score distribution on test sample\n{model_info}')
axes[0].legend()

# ROC
axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC curve\n{model_info}')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['BS', 'MC']).plot(ax=axes[2], colorbar=False)
axes[2].set_title(f'Confusion matrix (threshold=0.5)\n{model_info}')

plt.tight_layout()
plt.show()

In [ ]:
# Gem plot som PNG
save_png = BASE_DIR / 'results' / 'dynedge_mc_bs_fast_test_plots.png'
fig.savefig(save_png, dpi=150, bbox_inches='tight')
print('Saved:', save_png)